### Import libraries

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

### Load reference data and inspect columns to identify which columns to use

In [ ]:
# Load your reference AnnData object
adata = sc.read_h5ad("/home/ext_sana_noor_astraeabio_com/ext_hd/chiba/out_feb2025_xenium/script02a_gbmap_reduced/gbmap_reduced.h5ad")

# Inspect available cell type columns
print(adata.obs.columns)


Index(['author', 'donor_id', 'assay_ontology_term_id',
       'cell_type_ontology_term_id', 'development_stage_ontology_term_id',
       'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id',
       'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id',
       'annotation_level_1', 'annotation_level_2', 'annotation_level_3',
       'gbmap', 'method', 'stage', 'location', 'sector', 'celltype_original',
       'EGFR', 'MET', 'p53', 'TERT', 'ATRX', 'PTEN', 'MGMT', 'chr1p19q',
       'PDGFR', 'suspension_type', 'tissue_ontology_term_id', 'tissue_type',
       'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue',
       'self_reported_ethnicity', 'development_stage', 'observation_joinid'],
      dtype='object')


In [11]:
adata.var_names

Index(['ENSG00000069431', 'ENSG00000107796', 'ENSG00000151388',
       'ENSG00000145536', 'ENSG00000156140', 'ENSG00000120907',
       'ENSG00000170214', 'ENSG00000204305', 'ENSG00000204472',
       'ENSG00000171094',
       ...
       'ENSG00000107731', 'ENSG00000162692', 'ENSG00000038427',
       'ENSG00000165280', 'ENSG00000026025', 'ENSG00000146469',
       'ENSG00000174453', 'ENSG00000156076', 'ENSG00000169064',
       'ENSG00000184307'],
      dtype='object', length=356)

#### Set annotation level 3 for downstream analysis

In [ ]:
# Set annotation level 3 as the cell type column
cell_type_col = "annotation_level_3"

# Count cells per type
cell_counts = adata.obs[cell_type_col].value_counts()
print(cell_counts)


annotation_level_3
TAM-BDM            267733
TAM-MG             200606
AC-like            132265
OPC-like           118334
MES-like           105398
CD4/CD8            101945
Oligodendrocyte     49811
NPC-like            47578
Mono                41018
OPC                 25480
DC                   9124
Neuron               6309
Mural cell           6155
Endothelial          5819
Neutrophil           5117
RG                   4015
NK                   3151
B cell               2657
Plasma B             1543
Mast                  816
Astrocyte             803
Name: count, dtype: int64


### Filtering of cells
To make the data compatible for RCTD, we need to filter out cells types which have fewer than 25 cells. Next, we need to make sure the datatype is integers for counts data.  

In [12]:
# Filter out cell types with fewer than 25 cells
valid_cell_types = cell_counts[cell_counts >= 25].index.tolist()
adata_filtered = adata[adata.obs[cell_type_col].isin(valid_cell_types)].copy()

# Ensure gene names in .var["feature_name"]
if "feature_name" in adata_filtered.var:
    gene_names = adata_filtered.var["feature_name"]
else:
    gene_names = adata_filtered.var_names

# Convert to integer counts (required by RCTD)
counts = adata_filtered.X
if not isinstance(counts, np.ndarray):
    counts = counts.toarray()
counts = counts.astype(int)

# Make a DataFrame for counts: genes as rows, cells as columns
counts_df = pd.DataFrame(counts.T, index=gene_names, columns=adata_filtered.obs_names)

# Save counts to CSV (genes x cells)
counts_df.to_csv("/home/ext_sana_noor_astraeabio_com/ext_hd/chiba/rctd_data_prep/reference_for_rctd/sc_reference_counts.csv")

# Save cell type annotations: two columns (cell barcode, cell type)
celltypes_df = pd.DataFrame({
    "barcode": adata_filtered.obs_names,
    "cell_type": adata_filtered.obs[cell_type_col].values
})
celltypes_df.to_csv("/home/ext_sana_noor_astraeabio_com/ext_hd/chiba/rctd_data_prep/reference_for_rctd/sc_reference_celltypes.csv", index=False, header=False)